In [22]:
import json
import string
from nltk.corpus import stopwords

In [2]:
# Loading the lexicon in the RAM
with open('lexicon.json') as f:
    lexicon = json.load(f)

In [3]:
def generate_ngrams(word, n=3):
    return [word[i:i+n] for i in range(len(word) - n + 1)]

In [4]:
def jaccard_similarity(grams1, grams2):
    intersection = set(grams1).intersection(set(grams2))
    union = set(grams1).union(set(grams2))
    return len(intersection) / len(union)

In [5]:
def get_closest_match(query, lexicon, n=3):
    query_ngrams = generate_ngrams(query, n)
    best_match = None
    best_score = 0

    for word in lexicon:
        word_ngrams = generate_ngrams(word, n)
        score = jaccard_similarity(query_ngrams, word_ngrams)

        if score > best_score:
            best_score = score
            best_match = word
    
    return best_match

In [6]:
# Determining which barrel to search based on the first letter of the query
def get_barrel(query):
    first_letter = query[0]
    
    if 'a' <= first_letter <= 'c':
        return 'Barrels/barrel_a_c.json'
    
    elif 'd' <= first_letter <= 'h':
        return 'Barrels/barrel_d_h.json'
    
    elif 'i' <= first_letter <= 'm':
        return 'Barrels/barrel_i_m.json'
    
    elif 'n' <= first_letter <= 'r':
        return 'Barrels/barrel_n_r.json'
    
    elif 's' <= first_letter <= 'z':
        return 'Barrels/barrel_s_z.json'
    
    else:
        return None

In [10]:
def single_word_search(query, lexicon):
    query = query.lower()  # Normalizing the query

    if query not in lexicon:
        closest_match = get_closest_match(query, lexicon)
        print(f'No exact match found.\nInstead, showing results for {closest_match}')
        query = closest_match

    barrel_file = get_barrel(query)
    if barrel_file is None:
        return f"Word '{query}' not found in the inverted index"
    
    try:
        with open(barrel_file, 'r') as f:
            barrel = json.load(f)

            if query in barrel:
                return barrel[query]['postings']
            else:
                return f"Word '{query}' not found in the barrel file"
        
    except FileNotFoundError:
        return f"Barrel file '{barrel_file}' not found"
    
    return None

In [34]:
def multiple_word_search(query_string, lexicon):
    original_query = query_string

    # Removing punctuation marks from the query
    for char in query_string:
        if char in string.punctuation:
            query_string = query_string.replace(char, '')  # Removing punctuation marks from the query

    query_words = query_string.lower().split()  # Normalizing and then splitting the query
    if not query_words:
        return f"No result found for query: '{original_query}'\nQuery contains only punctuation marks."
    
    query_words = [word for word in query_words if word not in stopwords.words('english')]  
    if not query_words:
        return f"Query '{original_query}' not found in the lexicon."
    
    print(f"Query words: {query_words}")


    # Getting the list of documents for each word in the query
    posting_list = []

    # Fetch posting list for each word in the query
    for word in query_words:
        postings = single_word_search(word, lexicon)

        if postings is not None:
            posting_list.append(postings)  
    
    if not posting_list:
        return f"No result found for query: '{original_query}'"
    
    # Perform AND operation (intersection) on the posting lists
    intersection_docs = set(posting_list[0].keys())
    for postings in posting_list[1:]:
        intersection_docs.intersection_update(postings.keys())

    # Perform OR operation (union) on the posting lists if the result of intersection is very small
    union_docs = set()
    if len(intersection_docs) <= 3:

        for posting in posting_list:
            union_docs.update(posting.keys())
        
        # Excluding intersection documents from the union
        union_docs.difference_update(intersection_docs)

    
    # Preparing the result
    if(len(intersection_docs) <= 3):
        result = {
            'intersection': {
                "df": len(intersection_docs),
                "postings": sorted(intersection_docs, key = lambda doc: int(doc[3:]))
            } if intersection_docs else f"No common documents found for the query: '{original_query}'\nHere are some other results that might be relevant to the query: \n",

            'union': {
                "df": len(union_docs),
                "postings": sorted(union_docs, key = lambda doc: int(doc[3:]))
            } if union_docs else None
        }
    else:
        result = {
            'intersection': {
                "df": len(intersection_docs),
                "postings": sorted(intersection_docs, key = lambda doc: int(doc[3:]))
            }
        }


    return result

In [40]:
query = '#machine learnng'
print(multiple_word_search(query, lexicon))

Query words: ['machine', 'learnng']
No exact match found.
Instead, showing results for learn
{'intersection': {'df': 10650, 'postings': ['doc9', 'doc23', 'doc34', 'doc42', 'doc70', 'doc72', 'doc74', 'doc96', 'doc103', 'doc105', 'doc109', 'doc114', 'doc137', 'doc140', 'doc156', 'doc157', 'doc159', 'doc177', 'doc189', 'doc200', 'doc205', 'doc228', 'doc253', 'doc286', 'doc296', 'doc299', 'doc324', 'doc328', 'doc329', 'doc335', 'doc357', 'doc362', 'doc370', 'doc371', 'doc382', 'doc386', 'doc400', 'doc410', 'doc411', 'doc412', 'doc414', 'doc420', 'doc425', 'doc427', 'doc442', 'doc459', 'doc479', 'doc484', 'doc498', 'doc501', 'doc516', 'doc520', 'doc536', 'doc539', 'doc563', 'doc568', 'doc576', 'doc612', 'doc617', 'doc629', 'doc640', 'doc642', 'doc644', 'doc669', 'doc676', 'doc679', 'doc688', 'doc700', 'doc728', 'doc755', 'doc767', 'doc773', 'doc782', 'doc786', 'doc792', 'doc806', 'doc812', 'doc816', 'doc835', 'doc852', 'doc874', 'doc877', 'doc885', 'doc892', 'doc893', 'doc896', 'doc907', 'd

<hr>